# Spaceship Titanic — Complete EDA + ML Pipeline
**Goal:** Predict whether a passenger was `Transported` (True/False).  
**Metric:** Classification accuracy.

## 1. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

import lightgbm as lgb
import xgboost as xgb

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

SEED = 42
np.random.seed(SEED)

## 2. Load Data

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

print(f'Train shape: {train.shape}   Test shape: {test.shape}')
train.head()

## 3. EDA
### 3.1 Schema & Missing Values

In [ ]:
train.info()

In [ ]:
miss = pd.DataFrame({
    'train_missing': train.isnull().sum(),
    'train_%': (train.isnull().mean() * 100).round(2),
    'test_missing': test.isnull().sum(),
    'test_%': (test.isnull().mean() * 100).round(2)
})
display(miss[miss.train_missing > 0])

### 3.2 Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

vc = train['Transported'].value_counts()
axes[0].bar(vc.index.astype(str), vc.values, color=['steelblue', 'coral'])
axes[0].set_title('Transported counts')
axes[0].set_xlabel('Transported')
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontsize=11)

axes[1].pie(vc.values, labels=vc.index.astype(str), autopct='%1.1f%%',
            colors=['steelblue', 'coral'], startangle=90)
axes[1].set_title('Transported split')

plt.suptitle('Target Distribution (near-balanced)', fontsize=13)
plt.tight_layout()
plt.show()
print(vc)

### 3.3 Categorical Features vs Target

In [ ]:
cat_cols = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, col in zip(axes.flatten(), cat_cols):
    grp = (train.groupby(col)['Transported']
               .value_counts(normalize=True)
               .rename('pct')
               .reset_index())
    sns.barplot(data=grp, x=col, y='pct', hue='Transported', ax=ax)
    ax.set_title(f'{col} vs Transported rate')
    ax.set_ylabel('Proportion')
    ax.set_xlabel('')

plt.suptitle('Categorical Features vs Target', fontsize=14)
plt.tight_layout()
plt.show()

### 3.4 Spending Feature Distributions

In [ ]:
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

for ax, col in zip(axes.flatten(), spend_cols):
    for transported, grp in train.groupby('Transported'):
        vals = grp[col].dropna()
        ax.hist(np.log1p(vals), bins=40, alpha=0.6, label=str(transported))
    ax.set_title(f'log1p({col})')
    ax.legend(title='Transported')

ax = axes.flatten()[-1]
train['_TotalSpend'] = train[spend_cols].sum(axis=1)
for transported, grp in train.groupby('Transported'):
    ax.hist(np.log1p(grp['_TotalSpend']), bins=40, alpha=0.6, label=str(transported))
ax.set_title('log1p(TotalSpend)')
ax.legend(title='Transported')
train.drop(columns='_TotalSpend', inplace=True)

plt.suptitle('Spending Distributions by Transported (log scale)', fontsize=13)
plt.tight_layout()
plt.show()

### 3.5 Age Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for transported, grp in train.groupby('Transported'):
    axes[0].hist(grp['Age'].dropna(), bins=40, alpha=0.6, label=str(transported))
axes[0].set_title('Age distribution by Transported')
axes[0].legend(title='Transported')
axes[0].set_xlabel('Age')

sns.boxplot(data=train, x='Transported', y='Age', ax=axes[1], palette='muted')
axes[1].set_title('Age boxplot by Transported')

plt.tight_layout()
plt.show()

### 3.6 Correlation Heatmap

In [ ]:
num_cols = ['Age'] + ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
corr_df = train[num_cols + ['Transported']].copy()
corr_df['Transported'] = corr_df['Transported'].astype(int)

plt.figure(figsize=(9, 7))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix (numeric features)')
plt.tight_layout()
plt.show()

### 3.7 CryoSleep Spending Validation

In [ ]:
cryo = train[train['CryoSleep'] == True]
print('Spending while in CryoSleep (should be ~0):')
print(cryo[['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']].sum())
print(f'\nCryoSleep Transported rate     : {cryo["Transported"].mean():.3f}')
print(f'Non-CryoSleep Transported rate : {train[train["CryoSleep"]==False]["Transported"].mean():.3f}')

### 3.8 Cabin Deck Analysis

In [ ]:
train['_Deck'] = train['Cabin'].str.split('/').str[0]
deck_rate = (train.groupby('_Deck')['Transported']
                  .agg(['mean', 'count'])
                  .rename(columns={'mean': 'transport_rate', 'count': 'n'})
                  .sort_values('transport_rate', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
deck_rate['transport_rate'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Transported rate by Deck')
axes[0].axhline(0.5, color='red', linestyle='--', linewidth=1)
axes[0].set_ylabel('Rate')

deck_rate['n'].plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Passenger count by Deck')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()
train.drop(columns='_Deck', inplace=True)

## 4. Feature Engineering

In [ ]:
SPEND_COLS = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

def engineer(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # PassengerId → group + member_num
    split_id = df['PassengerId'].str.split('_', expand=True)
    df['Group']     = split_id[0].astype(int)
    df['MemberNum'] = split_id[1].astype(int)

    # Group size proxy
    group_size = df.groupby('Group')['MemberNum'].transform('count')
    df['GroupSize'] = group_size
    df['IsAlone']   = (group_size == 1).astype(int)

    # Cabin → Deck, CabinNum, Side
    cabin_split = df['Cabin'].str.split('/', expand=True)
    df['Deck']     = cabin_split[0]
    df['CabinNum'] = pd.to_numeric(cabin_split[1], errors='coerce')
    df['Side']     = cabin_split[2]

    # Fix cryo inconsistency: cryo passengers shouldn't spend
    cryo_mask = df['CryoSleep'] == True
    df.loc[cryo_mask, SPEND_COLS] = df.loc[cryo_mask, SPEND_COLS].fillna(0)

    # Spending aggregates
    df['TotalSpend'] = df[SPEND_COLS].sum(axis=1)
    df['HasSpend']   = (df['TotalSpend'] > 0).astype(int)
    df['LogSpend']   = np.log1p(df['TotalSpend'])

    # Age bins
    df['AgeBin'] = pd.cut(df['Age'],
                          bins=[0, 12, 18, 30, 50, 200],
                          labels=['child', 'teen', 'young_adult', 'adult', 'senior'])

    # Boolean → int
    for col in ['CryoSleep', 'VIP']:
        df[col] = df[col].map({True: 1, False: 0, 'True': 1, 'False': 0})

    return df


train_fe = engineer(train)
test_fe  = engineer(test)

print(f'After FE — train: {train_fe.shape}   test: {test_fe.shape}')
train_fe.head(3)

## 5. Preprocessing

In [ ]:
DROP_COLS = ['PassengerId', 'Name', 'Cabin', 'Group', 'MemberNum']
CAT_COLS  = ['HomePlanet', 'Destination', 'Deck', 'Side', 'AgeBin']
NUM_COLS  = ['Age', 'CabinNum', 'GroupSize', 'TotalSpend', 'LogSpend'] + SPEND_COLS
BIN_COLS  = ['CryoSleep', 'VIP', 'IsAlone', 'HasSpend']

def preprocess(df_fe, encoders=None, num_imputer=None, fit=True):
    df = df_fe.drop(columns=DROP_COLS, errors='ignore').copy()

    if fit:
        num_imputer = SimpleImputer(strategy='median')
        df[NUM_COLS] = num_imputer.fit_transform(df[NUM_COLS])
    else:
        df[NUM_COLS] = num_imputer.transform(df[NUM_COLS])

    if fit:
        encoders = {}
    for col in CAT_COLS:
        df[col] = df[col].astype(str).fillna('Unknown')
        if fit:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col])
            encoders[col] = le
        else:
            le = encoders[col]
            known = set(le.classes_)
            df[col] = df[col].apply(lambda x: x if x in known else 'Unknown')
            df[col] = le.transform(df[col])

    for col in BIN_COLS:
        df[col] = df[col].fillna(0).astype(int)

    feature_cols = NUM_COLS + CAT_COLS + BIN_COLS
    return df[feature_cols], encoders, num_imputer


X, encoders, num_imputer = preprocess(train_fe, fit=True)
y = train_fe['Transported'].astype(int)

X_test, _, _ = preprocess(test_fe, encoders=encoders, num_imputer=num_imputer, fit=False)

print(f'X: {X.shape}   X_test: {X_test.shape}')
X.head(3)

## 6. Model Benchmarking — 5-Fold CV

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

models = {
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=500, random_state=SEED))
    ]),
    'RandomForest': RandomForestClassifier(
        n_estimators=300, max_depth=10,
        min_samples_leaf=5, random_state=SEED, n_jobs=-1
    ),
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05,
        max_depth=5, random_state=SEED
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=400, learning_rate=0.05,
        max_depth=6, subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss', random_state=SEED, n_jobs=-1
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=500, learning_rate=0.05,
        num_leaves=63, min_child_samples=30,
        subsample=0.8, colsample_bytree=0.8,
        random_state=SEED, n_jobs=-1, verbose=-1
    ),
}

results = {}
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    results[name] = scores
    print(f'{name:<25} acc: {scores.mean():.4f} ± {scores.std():.4f}')

In [ ]:
means = {k: v.mean() for k, v in results.items()}
stds  = {k: v.std()  for k, v in results.items()}
names = list(means.keys())
accs  = [means[n] for n in names]
errs  = [stds[n]  for n in names]
max_acc = max(accs)
colors = ['coral' if a == max_acc else 'steelblue' for a in accs]

plt.figure(figsize=(10, 5))
bars = plt.bar(names, accs, yerr=errs, capsize=5, color=colors, edgecolor='white')
plt.ylim(0.75, 0.86)
plt.ylabel('CV Accuracy')
plt.title('5-Fold CV Accuracy Comparison')
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{acc:.4f}', ha='center', fontsize=9)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

best_model_name = max(means, key=means.get)
print(f'\nBest model: {best_model_name}  (acc={means[best_model_name]:.4f})')

## 7. Feature Importance (LightGBM)

In [ ]:
lgbm_model = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05,
    num_leaves=63, min_child_samples=30,
    subsample=0.8, colsample_bytree=0.8,
    random_state=SEED, n_jobs=-1, verbose=-1
)
lgbm_model.fit(X, y)

fi = pd.Series(lgbm_model.feature_importances_, index=X.columns)
fi_sorted = fi.sort_values(ascending=True)

plt.figure(figsize=(9, 7))
fi_sorted.tail(20).plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Top-20 Feature Importances (LightGBM)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 8. Hyperparameter Tuning (LightGBM)

In [ ]:
param_grid = {
    'num_leaves':        [63, 127],
    'min_child_samples': [20, 40],
    'learning_rate':     [0.03, 0.05],
    'colsample_bytree':  [0.7, 0.9],
}

grid_model = lgb.LGBMClassifier(
    n_estimators=500, subsample=0.8,
    random_state=SEED, n_jobs=-1, verbose=-1
)

gs = GridSearchCV(
    grid_model, param_grid,
    cv=cv, scoring='accuracy',
    n_jobs=-1, verbose=1
)
gs.fit(X, y)

print(f'Best params : {gs.best_params_}')
print(f'Best CV acc : {gs.best_score_:.4f}')

## 9. Final Model — Validation Metrics

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

final_model = lgb.LGBMClassifier(
    n_estimators=700, subsample=0.8,
    random_state=SEED, n_jobs=-1, verbose=-1,
    **gs.best_params_
)
final_model.fit(X_tr, y_tr)

y_pred  = final_model.predict(X_val)
y_proba = final_model.predict_proba(X_val)[:, 1]

print(f'Validation Accuracy : {accuracy_score(y_val, y_pred):.4f}')
print(f'Validation ROC-AUC  : {roc_auc_score(y_val, y_proba):.4f}')
print()
print(classification_report(y_val, y_pred, target_names=['Not Transported', 'Transported']))

In [ ]:
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Transported', 'Transported'],
            yticklabels=['Not Transported', 'Transported'])
plt.title('Confusion Matrix — Validation Set')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 10. Retrain on Full Data & Generate Submission

In [ ]:
final_full = lgb.LGBMClassifier(
    n_estimators=700, subsample=0.8,
    random_state=SEED, n_jobs=-1, verbose=-1,
    **gs.best_params_
)
final_full.fit(X, y)

test_preds_bool = final_full.predict(X_test).astype(bool)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': test_preds_bool
})
submission.to_csv('submission.csv', index=False)

print(f'Submission saved: {submission.shape[0]} rows')
print(submission['Transported'].value_counts())
submission.head()

## 11. Key Insights

| Finding | Detail |
|---------|--------|
| **Balanced target** | ~50.4% Transported — no class-weight adjustments needed |
| **CryoSleep is king** | Passengers in cryo-sleep transported at ~82% vs ~43% otherwise |
| **Deck matters** | Decks B/C (Europa) have higher transport rates; G/T lower |
| **Spending signals** | Zero spend + CryoSleep=False is unusual; spending negatively correlates with Transported |
| **HomePlanet** | Europa passengers transported more than Earth/Mars |
| **Best model** | LightGBM with tuned hyperparameters (~80-81% CV accuracy) |
| **Top features** | CryoSleep, TotalSpend/LogSpend, Spa, VRDeck, Deck |